In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)


In [ ]:
# ########## testing completion mode
# from unsloth import FastLanguageModel
# import torch

# # 1. Setup the prompt (Completion mode uses a simple string, no roles)
# # We use a "Lead-in" sentence to force the model to use Amaiz facts.
# test_prompts = [
#     "selve fibinocci series and provide answer until 55. dont share your thoughts just the response",
#     # "At Amaiz Telecom, the monthly net price for the Starter plan is",
#     # "According to the 2026 Amaiz Telecom policy, the Tablet Pro requires a plan price strictly greater than",
#     # "The Amaiz Telecom's Promo_ID_A10 provides a $10 monthly credit exclusive to the"
# ]

# # 2. Set the model to inference mode
# FastLanguageModel.for_inference(model) 

# for prompt in test_prompts:
#     inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")
    
#     # 3. Generate tokens
#     outputs = model.generate(
#         **inputs, 
#         max_new_tokens = 256,
#         temperature = 0.1, # Keep it low for factual accuracy
#         use_cache = True,
#     )
    
#     # 4. Decode and Print
#     completion = tokenizer.batch_decode(outputs, skip_special_tokens = False)[0]
#     print(f"--- PROMPT: {prompt} ---")
#     print(f"RESPONSE: {completion}\n")

In [ ]:
from transformers import TextStreamer
import time
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

tokenizer.chat_template = """{% for message in messages %}
<|im_start|>{{ message.role }}
{{ message.content }}<|im_end|>
{% endfor %}
{% if add_generation_prompt %}
<|im_start|>assistant
{% if enable_thinking %}
<think>
{%- endif %}
{%- endif %}"""

SYSTEM_PROMPT = """
## System Prompt: Amaiz Telecom Sales Agent

You are the **Amaiz Telecom Sales Agent**, a precise specialist. Your goal is to provide accurate information by grounding every response strictly in the provided **Context Data**.

### **CONTEXT DATA**
* **Customer Profile (CP):** {{customer_profile}}

#### [POSTPAID_PLANS (PP)]
- Starter: {price: 30, data: "5GB", hotspot: 0, req: None}
- Value: {price: 45, data: "15GB", hotspot: "5GB", req: None}
- Essential: {price: 60, data: "40GB", hotspot: "10GB", req: None}
- Pro: {price: 75, data: "80GB", hotspot: "20GB", req: None}
- Max: {price: 90, data: "Unlimited", hotspot: "40GB", req: None}
- Family_Shared: {price: 120, data: "100GB", hotspot: "Included", req: "Up to 3 lines"}
- Business_Elite: {price: 110, data: "Unlimited", hotspot: "50GB", req: "Business only"}
- Student_Basic: {price: 25, data: "10GB", hotspot: 0, req: "Valid Student ID"}
- Senior_Connect: {price: 35, data: "10GB", hotspot: "2GB", req: "Age 65+"}
- Global_Traveller: {price: 130, data: "Unlimited", hotspot: "5GB Intl", req: "Roaming"}

#### [DEVICE_BUNDLES (DB)] (24-mo)
- Phone_X1: {cost: 20, min_plan: "Essential+"}
- Phone_Z5: {cost: 35, min_plan: "Max/Pro"}
- Tablet_Pro: {cost: 15, min_plan: "Price > 60"}
- Budget_A1: {cost: 5, min_plan: "Starter/Value"}
- Watch_Series: {cost: 10, min_plan: "Any active voice"}

#### [PROMOTIONS (PR)]
- Promo_A10: {value: -10, duration: "12mo", target: "Max, Pro"}
- Promo_B20: {value: -20, duration: "3mo", target: "Family_Shared"}
- Promo_C5: {value: -5, type: "Auto-Pay", target: "All except Student_Basic"}

#### [RULES_DISCLAIMERS (RD)]
- Auto-Pay: Check the 'auto_Pay' field in the Customer Profile (CP) first. If 'true', use the listed price. If 'false', explicitly state the $5 increase in your Transparent Logic section. Users might use auto_pay or its synonyms in their question and treat all same.
- Taxes: Prices exclude gov taxes/fees. DO NOT calculate or estimate tax amounts in the total unless a specific percentage is provided.
- Credit: Device bundles require credit approval and 24-mo contract.
- Tablet_Pro_Check: Eligibility is based on the UN-DISCOUNTED Base Plan Price > $60. Eligibility for Tablet_Pro requires a base plan price EXCLUSIVELY and STRICTLY GREATER than $60.00. A plan priced at exactly $60.00 is INELIGIBLE and does not meet the threshold.
---

### **INSTRUCTION CODES (IC)**
Execute logic based on the codes provided in the user request:
* **[[AMZT_MA]]**: Perform and state explicit mathematical calculations (e.g., Base Plan + Device Cost = Total) based on context fields.
* **[[AMZT_SP]]**: Recommend products or plans driven by **User Intent** and **Customer Profile**. 
    - **Priority**: User Intent takes higher priority than the profile.
    - **Conflict Handling**: If Intent conflicts with Profile, quote the Intent-based price and explicitly mention the loss of profile-based discounts in your explanation.
    - **Verification**: For restricted plans, you must explicitly state the Customer Profile (CP) requirement found in the context and verify if the user meets it before recommending.
* **[[AMZT_LG]]**: Retrieve and include specific Disclaimer text related to the discussed items.
* **[[AMZT_FQ]]**: Provide a definitive conclusion, value, or total price.

---

### **RESPONSE GUIDELINES**
1. **Logical Narrative Flow (Mandatory):** To ensure accuracy and grounding, every response must follow this specific sequence within a natural conversation (do not use "Step" labels):
    * **Acknowledge Current Context:** Begin by identifying the customer's current plan and its price from the provided Profile.
    * **Transparent Logic & Math:** Explain the specific rule, eligibility threshold, or promotion being applied. Show the explicit addition or subtraction (e.g., "$75 plan + $15 device") to justify the result. Stick strictly to the prices listed in CONTEXT DATA. Do not invent, estimate, or assume promotional pricing that is not explicitly stated for a specific plan tier.
    * **Clear Conclusion:** Conclude with the definitive recommendation and the final total monthly cost.
2. **Grounding & Out-of-Domain**: 
    * If the request is **out-of-domain**, state clearly that as an Amaiz Specialist, you focus exclusively on Amaiz telecom products. Do not provide information on unrelated topics.
    * If the request is **in-domain** but data is missing, offer the closest available alternative found in the context.
3. **Format**: **Do not use `<t>` tags or "Step X" headers.** Use clear, professional, and conversational language. The response must be a single, cohesive explanation that prioritizes factual accuracy through sequential reasoning.

"""

import json
import json

# Define your 4 Evaluation Profiles
EVAL_PROFILES = {
    "STUDENT": { "age": 19, "is_student": True, "is_business": False, "auto_debit": True, "avg_usage_gb": 8.5, "current_plan": "Student_Basic"},
    "SENIOR": { "age": 73, "is_student": False, "is_business": False, "auto_debit": True, "avg_usage_gb": 4, "current_plan": "Starter"},
    "LOGIC_TRAP": { "age": 29, "is_student": False, "is_business": False, "auto_debit": True, "avg_usage_gb": 13, "current_plan": "Essential"},
    "STANDARD_HIGH": { "age": 46, "is_student": False, "is_business": False, "auto_debit": True, "avg_usage_gb": 77, "current_plan": "Max"},
    "BUSINESS": { "age": 28, "is_student": False, "is_business": True, "auto_debit": True, "avg_usage_gb": 99, "current_plan": "Business_Elite"}
}


# Define your Evaluation Questions
QUESTIONS = [
    # --- Category: Mathematical & Promotion Accuracy ([[AMZT_MA]]) ---
    # "I see the Scholar_Basic plan is $20. Since I don't want to use Auto-Debit, what will my total monthly price be for just the plan?",
    "I'm interested in adding the Tablet_Pro to my current setup. Can you check if my current plan qualifies for this device bundle? If not, what is the cheapest plan I can switch to that allows me to get the tablet, and what would my new monthly total be (assuming I keep auto-debit)?"
    # "What is my total cost for the Infinite plan with a Titan_V1 device using Auto-Debit?",
    # "If I turn off Auto-Debit on the Ultra plan, what will my new monthly total be?",
    # "I want the Standard plan but I will pay manually. How much is the monthly charge?",
    # "Calculate the total for an Infinite plan and a Smart_Band with Auto-Debit enabled.",
    
    # # --- Category: Hardware Logical Traps (RD.3 & RD.5) ---
    # "I want to add the Tab_Air to my Premium plan. Is that possible?",
    # "Can I get the Tab_Air deal if I switch to the Ultra plan?",
    # "What is the cheapest plan I can use to qualify for the Titan_V1 bundle?",
    # "Does the Lite plan qualify for the Smart_Band bundle? What is the total cost?",
    # "If I am on the Standard plan, am I eligible for the Tab_Air?",

    # # --- Category: Product Recommendation & Eligibility ([[AMZT_SP]]) ---
    # "I'm over 60. Do I qualify for a cheaper plan? What is the price?",
    # "I'm starting a business and need unlimited data. Can I get the Biz_Pro plan?",
    # "I use about 30GB of data a month. Which plan fits my needs best?",
    # "I am a student. What is the best plan you have for me and how much data does it include?",
    # "I need the maximum amount of hotspot data available. Which plan should I pick?",

    # # --- Category: Out-of-Domain & Guardrails (OOD) ---
    # "How does Amaiz Telecom coverage compare to Verizon or AT&T?",
    # "Can you help me troubleshoot my internet connection? It's very slow today.",
    # "What are the best-selling smartphones at Best Buy right now?",
    # "I want to bake a cake for my friend's birthday, do you have a good recipe?",
    # "Is Amaiz Telecom planning to release any new 5G plans next month?"
]

OUTPUT_FILE = "/mnt/data/training_data/evaluation_results.jsonl"

# --- RUN EVALUATION ---
with open(OUTPUT_FILE, "w") as f:
    for profile_name, profile_json in EVAL_PROFILES.items():
        print(f"\n--- Testing Profile: {profile_name} ---")
        
        for question in QUESTIONS:
            # 1. Prepare Prompt
            system_prompt_enriched = SYSTEM_PROMPT.replace("{{customer_profile}}", json.dumps(profile_json))
            messages = [
                {"role": "system", "content": system_prompt_enriched},
                {"role": "user", "content": question}
            ]

            # 2. Tokenize and Generate
            inputs = tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_tensors="pt"
            ).to("cuda")

            # Capture input token count (The size of your system prompt + question)
            input_token_count = inputs['input_ids'].shape[1]

            # Start the timer
            start_time = time.time()

            # Execute model generation
            outputs = model.generate(
                input_ids=inputs,
                max_new_tokens=256,
                do_sample=False,
                temperature=0.0,
                repetition_penalty=1.05,
                eos_token_id=tokenizer.eos_token_id
            )

            # Calculate total response time
            total_response_time = time.time() - start_time
            
            # 3. Decode Response and Capture Output Token Count
            # Slicing the output [len(inputs[0]):] ensures we only count NEW tokens
            generated_tokens = outputs[0][len(inputs[0]):]
            output_token_count = len(generated_tokens)
            
            response_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
            print(f"Q: {question}\nA: {response_text[:100]}...") # Console preview

            # 4. Save to JSONL with Performance Metrics
            eval_entry = {
                "profile_name": profile_name,
                "profile_data": profile_json,
                "question": question,
                "model_response": response_text,
                "metrics": {
                    "input_tokens": int(input_token_count),
                    "output_tokens": int(output_token_count),
                    "total_latency_sec": round(total_response_time, 4),
                    "tokens_per_sec": round(output_token_count / total_response_time, 2) if total_response_time > 0 else 0
                }
            }
            f.write(json.dumps(eval_entry) + "\n")

print(f"\nEvaluation Complete. Results saved to {OUTPUT_FILE}")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],

                      # "embed_tokens", "lm_head",], # Add for continual pretraining
    lora_alpha = 32,
    lora_dropout = 0.1, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = True,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)


In [ ]:
# from unsloth import UnslothTrainer, UnslothTrainingArguments
# from datasets import load_dataset, Dataset

# with open("/mnt/data/training_data/cpt_060326.txt", "r", encoding="utf-8") as f:
#     raw_text = f.read()

# EOS_TOKEN=tokenizer.eos_token

# cpt_dataset = Dataset.from_dict({"text": [raw_text+EOS_TOKEN] * 50})

# def format_cpt_text(examples):
#     return {"text": examples["text"]}

# cpt_dataset = cpt_dataset.map(format_cpt_text, batched=True)

# trainer = UnslothTrainer(
#     model = model,
#     tokenizer = tokenizer,
#     train_dataset = cpt_dataset,
#     dataset_text_field = "text",
#     # formatting_func = format_cpt_text,
#     max_seq_length = max_seq_length,
#     dataset_num_proc = 2,
#     packing = True,
#     dataset_batched=True,
#     args = UnslothTrainingArguments(
#         per_device_train_batch_size = 2,
#         gradient_accumulation_steps = 4,
#         warmup_steps = 10,
#         num_train_epochs = 5,
#         learning_rate = 5e-5,
#         logging_steps = 5,
#         optim = "adamw_8bit",
#         weight_decay = 0.1,
#         packing = True,
#         lr_scheduler_type = "cosine",
#         seed = 3407,
#         output_dir = "outputs",
#         report_to = "none"
#     ),
# )
# trainer_stats = trainer.train()

In [ ]:
#save CPT
# model.save_pretrained("/mnt/data/models/cpt_lora_adap_060326_v1")
# tokenizer.save_pretrained("/mnt/data/models/cpt_lora_adap_060326_v1")

In [ ]:
######### data set for SFT

import json
from datasets import Dataset

tokenizer.chat_template = """{% for message in messages %}
<|im_start|>{{ message.role }}
{{ message.content }}<|im_end|>
{% endfor %}
{% if add_generation_prompt %}
<|im_start|>assistant
{% if enable_thinking %}
<think>
{%- endif %}
{%- endif %}"""

def prepare_data_set(file_name):
    sft_data = []

    # 1. Open the file and read it line by line
    "/mnt/data/training_data/amaiz_telecom_train_v1.jsonl"
    with open(file_name, "r", encoding="utf-8") as f:
        for line in f:
            # Clean up any empty lines
            if line.strip():
                # Convert the string line into a Python Dictionary
                sft_data.append(json.loads(line))

    sft_dataset = Dataset.from_dict({
    "messages": [item["messages"] for item in sft_data]
    })

    print(f"Dataset loaded with {len(sft_dataset)} individual rows.")
    # print(sft_dataset[0])

    def formatting_prompts_func(examples):
        output_texts = []
        for i in range(len(examples['messages'])):
            # apply_chat_template converts the list of dicts into a single string
            text = tokenizer.apply_chat_template(examples['messages'][i], tokenize=False, add_generation_prompt=False)
            if not text.endswith(tokenizer.eos_token):
                text += tokenizer.eos_token
            output_texts.append(text)
        return { "text" : output_texts }

    # Map the dataset to the new format
    sft_dataset = sft_dataset.map(formatting_prompts_func, batched=True)

    print(sft_dataset[0]['text'])

    def tokenize_function(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            max_length=max_seq_length,
            padding=False # Collator handles padding
        )

    # Apply tokenization and REMOVE the text columns
    tokenized_dataset = sft_dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=sft_dataset.column_names # This removes 'messages' and 'text'
    )
    return tokenized_dataset


In [ ]:
### SFT trainer initialization
from unsloth import UnslothTrainer, UnslothTrainingArguments
from transformers import DataCollatorForLanguageModeling

response_template = "<|im_start|>assistant\n"

class ManualCompletionCollator(DataCollatorForLanguageModeling):
    def __init__(self, response_template, tokenizer, *args, **kwargs):
        super().__init__(tokenizer, *args, mlm=False, **kwargs)
        self.response_template = response_template
        self.response_token_ids = tokenizer.encode(response_template, add_special_tokens=False)

    def torch_call(self, examples):
        batch = super().torch_call(examples)
        
        for i in range(len(batch["labels"])):
            response_token_ids_start_idx = None
            
            # Find the index where the assistant response starts
            for idx in range(len(batch["labels"][i])):
                if batch["labels"][i][idx : idx + len(self.response_token_ids)].tolist() == self.response_token_ids:
                    response_token_ids_start_idx = idx + len(self.response_token_ids)
                    break

            if response_token_ids_start_idx is not None:
                # Mask everything before the assistant response starts
                batch["labels"][i][:response_token_ids_start_idx] = -100
                
        return batch

# Initialize it
collator = ManualCompletionCollator(
    response_template="<|im_start|>assistant\n", 
    tokenizer=tokenizer
)

# split_dataset = tokenized_dataset.train_test_split(test_size=0.2, seed=42)

# Access them individually
train_dataset = prepare_data_set("/mnt/data/training_data/amaiz_telecom_train_v1.jsonl")
eval_dataset = prepare_data_set("/mnt/data/training_data/amaiz_telecom_valid_v1.jsonl")

sft_trainer = UnslothTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    data_collator=collator,
    # dataset_text_field = "text",
    # formatting_func = format_sft_custom,
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    dataset_batched=True,
    args = UnslothTrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        logging_steps = 5,
        save_steps = 10,
        save_total_limit=3,
        eval_strategy="steps",  # Tell the trainer to evaluate periodically
        eval_steps=10,                # Run evaluation every 10 steps
        metric_for_best_model="loss", # Or "eval_loss" (lower is better)
        greater_is_better=False,
        load_best_model_at_end=True,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "/mnt/data/outputs/",
        report_to = "none",
        completion_only_loss=True
    ),
)

## SFT training


In [ ]:

# # 1. Grab 2 tokenized samples
# samples = [tokenized_dataset[i] for i in range(2)]

# # 2. Run through collator (it now finds input_ids!)
# batch = collator(samples)

# # 3. Check Labels
# for i in range(len(samples)):
#     labels = batch["labels"][i].tolist()
#     # Decode only the tokens that aren't -100
#     decoded_labels = tokenizer.decode([t for t in labels if t != -100])
    
#     print(f"\n--- Sample {i+1} Unmasked Content ---")
#     print(decoded_labels)

In [ ]:
sft_trainer.train()

In [ ]:
model.save_pretrained("/mnt/data/models/amaiz_telecom_model_4b_0903_adap")
tokenizer.save_pretrained("/mnt/data/models/amaiz_telecom_model_4b_0903_adap")

In [ ]:
model.save_pretrained_gguf("/mnt/data/models/amaiz_telecom_gguf", tokenizer, quantization_method = "q4_k_m")

In [ ]:
for name, param in model.named_parameters():
    if "lora" in name:
        if not param.requires_grad:
            print(f"🚨 WARNING: {name} is FROZEN! Training will not work.")
        else:
            print(f"✅ {name} is trainable.")
        break